# YouTube Music History Analytics

This notebook retrieves the listening history available to the authenticated YouTube Music account for `@staceymaples`, exports the raw history using the KZSU playlist CSV columns, and produces ranked track, artist, and monthly reports.

YouTube Music history is private. The public handle labels the report, while the local browser-authentication file provides access to the matching account.

## 1. Setup – imports and configuration

In [1]:
import json
from pathlib import Path

import ytmusicapi

# VS Code may execute this notebook with notebooks/ as the working directory.
WORKSPACE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

# Convert the saved JSON header object into the line format expected by setup().
header_candidates = [
    WORKSPACE_DIR / 'src/dj_stace/scratch.json',
    WORKSPACE_DIR / 'scratch.json',
    Path('scratch.json'),
]
header_path = next((path for path in header_candidates if path.exists()), None)
if header_path is None:
    raise FileNotFoundError('Could not find scratch.json in the workspace.')

header_values = json.loads(header_path.read_text(encoding='utf-8'))
headers_raw = '\n'.join(f'{key}: {value}' for key, value in header_values.items())
yt_music_config = ytmusicapi.setup(
    filepath=str(WORKSPACE_DIR / 'browser.json'),
    headers_raw=headers_raw,
)
print(f'Created {WORKSPACE_DIR / "browser.json"} from saved request headers.')

Created /Users/maples/Github/KZSU-Library-Show/browser.json from saved request headers.


In [2]:
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

# Install the maintained YouTube Music client into the active notebook environment if needed.
try:
    from ytmusicapi import YTMusic
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ytmusicapi'])
    from ytmusicapi import YTMusic

WORKSPACE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUTPUT_DIR = WORKSPACE_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

# Authentication is intentionally kept outside source control.
# The ytmusicapi browser command creates browser.json by default.
AUTH_PATH = Path(os.getenv('YTMUSIC_AUTH_PATH', str(WORKSPACE_DIR / 'browser.json')))


## 2. Account and output parameters

In [3]:
# YouTube Music handle used for labeling and output names.
YTM_USER_HANDLE = '@staceymaples'

# The authenticated account behind AUTH_PATH must be the account represented by this handle.
# YouTube Music history is private and cannot be retrieved from a public handle alone.
HISTORY_OUTPUT_PATH = OUTPUT_DIR / 'ytm_staceymaples_history.csv'
TOP_TRACKS_OUTPUT_PATH = OUTPUT_DIR / 'ytm_staceymaples_top_100_tracks.csv'
TOP_ARTISTS_OUTPUT_PATH = OUTPUT_DIR / 'ytm_staceymaples_top_50_artists.csv'
MONTHLY_OUTPUT_PATH = OUTPUT_DIR / 'ytm_staceymaples_monthly_top_10.csv'
REPORT_OUTPUT_PATH = OUTPUT_DIR / 'ytm_staceymaples_analytics_report.md'

print(f'Preparing YouTube Music history retrieval for {YTM_USER_HANDLE}')
print(f'Authentication file: {AUTH_PATH}')

Preparing YouTube Music history retrieval for @staceymaples
Authentication file: /Users/maples/Github/KZSU-Library-Show/browser.json


## 3. Retrieve authenticated YouTube Music history

In [4]:
def load_ytmusic_client(auth_path):
    """Create an authenticated YouTube Music client from browser headers."""
    if not auth_path.exists():
        raise FileNotFoundError(
            f'Authentication file not found: {auth_path}. '
            'Run "ytmusicapi browser" and save the generated file there.'
        )

    ytm = YTMusic(str(auth_path))
    account_info = ytm.get_account_info()
    actual_handle = account_info.get('channelHandle', '')
    if actual_handle and actual_handle.lower() != YTM_USER_HANDLE.lower():
        raise ValueError(
            f'Authenticated account is {actual_handle}, not {YTM_USER_HANDLE}. '
            'Create the browser auth file while signed in to the intended account.'
        )
    return ytm


def _artist_text(artists):
    if not artists:
        return ''
    return ', '.join(
        artist.get('name', '') if isinstance(artist, dict) else str(artist)
        for artist in artists
    )


def _album_text(album):
    if isinstance(album, dict):
        return album.get('name', '')
    return str(album or '')


def fetch_ytmusic_history(auth_path):
    """Fetch the history currently exposed by the authenticated account."""
    ytm = load_ytmusic_client(auth_path)
    history = ytm.get_history()
    if not history:
        raise RuntimeError(
            'YouTube Music returned no history. Confirm that the authenticated account '
            'is @staceymaples and that watch history is enabled.'
        )
    return history


def history_to_dataframe(history):
    """Normalize YouTube Music history to the KZSU playlist CSV convention."""
    rows = []
    for item in history:
        if item.get('videoId') is None and item.get('playlistId') is None:
            continue
        rows.append({
            'date': item.get('played', '') or item.get('date', ''),
            'time': item.get('time', ''),
            'show_name': 'YouTube Music history',
            'dj_name': YTM_USER_HANDLE,
            'artist': _artist_text(item.get('artists', [])),
            'title': item.get('title', ''),
            'album': _album_text(item.get('album', '')),
            'label': item.get('label', ''),
            'duration': item.get('duration', ''),
        })

    columns = ['date', 'time', 'show_name', 'dj_name', 'artist', 'title', 'album', 'label', 'duration']
    return pd.DataFrame(rows, columns=columns)


HISTORY_COLUMNS = ['date', 'time', 'show_name', 'dj_name', 'artist', 'title', 'album', 'label', 'duration']
if AUTH_PATH.exists():
    history_items = fetch_ytmusic_history(AUTH_PATH)
    history_df = history_to_dataframe(history_items)
    history_df.to_csv(HISTORY_OUTPUT_PATH, index=False)
    print(f'Retrieved {len(history_df)} history entries from YouTube Music')
    print(f'History CSV saved to {HISTORY_OUTPUT_PATH}')
else:
    history_df = pd.DataFrame(columns=HISTORY_COLUMNS)
    print(f'Authentication file not found: {AUTH_PATH}')
    print('Run "ytmusicapi browser" while signed in to YouTube Music, then rerun this cell.')

KeyError: "Unable to find 'header' using path ['actions', 0, 'openPopupAction', 'popup', 'multiPageMenuRenderer', 'header', 'activeAccountHeaderRenderer', 'accountName', 'runs', 0, 'text'] on {'sections': [{'multiPageMenuSectionRenderer': {'items': [{'compactLinkRenderer': {'icon': {'iconType': 'UNLIMITED'}, 'title': {'runs': [{'text': 'Get Music Premium'}]}, 'navigationEndpoint': {'clickTrackingParams': 'CAcQmbcBGAAiEwjlta-xm4iXAxWr6-MHHcTDGOrKAQRZteCL', 'browseEndpoint': {'browseId': 'SPunlimited', 'params': 'kgMXShUIBhIRdW5saW1pdGVkLUItbXVzaWM%3D', 'canonicalBaseUrl': '/music_premium'}}, 'trackingParams': 'CAcQmbcBGAAiEwjlta-xm4iXAxWr6-MHHcTDGOo='}}, {'compactLinkRenderer': {'icon': {'iconType': 'SETTINGS'}, 'title': {'runs': [{'text': 'Settings'}]}, 'navigationEndpoint': {'clickTrackingParams': 'CAYQ6qgHGAEiEwjlta-xm4iXAxWr6-MHHcTDGOrKAQRZteCL', 'applicationSettingsEndpoint': {'hack': True}}, 'trackingParams': 'CAYQ6qgHGAEiEwjlta-xm4iXAxWr6-MHHcTDGOo='}}, {'compactLinkRenderer': {'

## 4. Rank tracks, artists, and monthly listening

In [26]:
def build_ytmusic_reports(history_df):
    """Create ranked track, artist, monthly, and markdown reports from history."""
    if history_df.empty:
        print('No YouTube Music history loaded; reports will be created after authentication.')
        return (
            pd.DataFrame(columns=['rank', 'artist', 'title', 'plays']),
            pd.DataFrame(columns=['rank', 'artist', 'plays']),
            pd.DataFrame(columns=['month', 'rank', 'artist', 'title', 'plays']),
        )

    track_groups = history_df.groupby(
        ['artist', 'title', 'album', 'label', 'duration'],
        dropna=False,
    ).size().reset_index(name='plays')
    top_tracks = track_groups.sort_values(
        ['plays', 'artist', 'title'], ascending=[False, True, True]
    ).head(100).reset_index(drop=True)
    top_tracks.insert(0, 'rank', range(1, len(top_tracks) + 1))
    top_tracks.to_csv(TOP_TRACKS_OUTPUT_PATH, index=False)

    artist_groups = history_df.groupby('artist', dropna=False).size().reset_index(name='plays')
    top_artists = artist_groups.sort_values(
        ['plays', 'artist'], ascending=[False, True]
    ).head(50).reset_index(drop=True)
    top_artists.insert(0, 'rank', range(1, len(top_artists) + 1))
    top_artists.to_csv(TOP_ARTISTS_OUTPUT_PATH, index=False)

    dated = history_df.copy()
    dated['parsed_date'] = pd.to_datetime(dated['date'], errors='coerce')
    dated = dated.dropna(subset=['parsed_date']).copy()
    if dated.empty:
        monthly_top = pd.DataFrame(columns=['month', 'rank', 'artist', 'title', 'plays'])
    else:
        dated['month'] = dated['parsed_date'].dt.to_period('M').astype(str)
        monthly_groups = dated.groupby(['month', 'artist', 'title']).size().reset_index(name='plays')
        monthly_top = (
            monthly_groups.sort_values(
                ['month', 'plays', 'artist', 'title'],
                ascending=[True, False, True, True],
            )
            .groupby('month', group_keys=False)
            .head(10)
            .copy()
        )
        monthly_top['rank'] = monthly_top.groupby('month').cumcount() + 1
        monthly_top = monthly_top[['month', 'rank', 'artist', 'title', 'plays']]
    monthly_top.to_csv(MONTHLY_OUTPUT_PATH, index=False)

    report_lines = [
        '# YouTube Music history analytics',
        '',
        f'Account label: {YTM_USER_HANDLE}',
        f'History entries retrieved: {len(history_df)}',
        '',
        '## Top 100 tracks by plays',
    ]
    report_lines.extend(
        f"{row.rank}. {row.artist} — {row.title} ({row.plays})"
        for row in top_tracks.itertuples()
    )
    report_lines.extend(['', '## Top 50 artists by plays'])
    report_lines.extend(
        f"{row.rank}. {row.artist} ({row.plays})"
        for row in top_artists.itertuples()
    )
    report_lines.extend(['', '## Monthly top 10 tracks'])
    for month, month_rows in monthly_top.groupby('month', sort=True):
        entries = ', '.join(
            f'{row.artist} — {row.title} ({row.plays})'
            for row in month_rows.itertuples()
        )
        report_lines.append(f'- {month}: {entries}')
    REPORT_OUTPUT_PATH.write_text('\n'.join(report_lines) + '\n')

    print(f'Top tracks CSV saved to {TOP_TRACKS_OUTPUT_PATH}')
    print(f'Top artists CSV saved to {TOP_ARTISTS_OUTPUT_PATH}')
    print(f'Monthly top 10 CSV saved to {MONTHLY_OUTPUT_PATH}')
    print(f'Markdown report saved to {REPORT_OUTPUT_PATH}')
    return top_tracks, top_artists, monthly_top


top_tracks_df, top_artists_df, monthly_top_df = build_ytmusic_reports(history_df)

Top tracks CSV saved to /Users/maples/Github/KZSU-Library-Show/outputs/ytm_staceymaples_top_100_tracks.csv
Top artists CSV saved to /Users/maples/Github/KZSU-Library-Show/outputs/ytm_staceymaples_top_50_artists.csv
Monthly top 10 CSV saved to /Users/maples/Github/KZSU-Library-Show/outputs/ytm_staceymaples_monthly_top_10.csv
Markdown report saved to /Users/maples/Github/KZSU-Library-Show/outputs/ytm_staceymaples_analytics_report.md


/var/folders/fn/tgnb7dgn3sv29jhwftd9fgj40000gr/T/ipykernel_48444/4131585219.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dated['parsed_date'] = pd.to_datetime(dated['date'], errors='coerce')


## Notes and authentication

- The `@staceymaples` handle labels the outputs; it does not grant access to private history.
- Run `ytmusicapi browser` while signed in to the matching YouTube Music account; it creates `browser.json` by default.
- YouTube Music exposes the history currently available to that authenticated account; deleted or unavailable history cannot be reconstructed.
- The history export follows the KZSU columns: `date,time,show_name,dj_name,artist,title,album,label,duration`.
- Generated reports are saved under `outputs/`.